# TG 7-to-1 Distillation — No-Injection Verification Circuit

Simplified version of the full TG distillation that replaces magic state teleportation
with direct fold-transversal S gates:
- W1..W7: fold-transversal S  (simulates perfect |Y> teleportation outcome)
- W0: fold-transversal S_dag  (output correction)

All qubits measured in X basis.

## Circuit steps
1. Initialize W0-W7 (same as full distillation: W0/W1/W2/W4 in |+>, W3/W5/W6/W7 in |0>)
2. d rounds SE (initialization)
3. Hypercube CNOT encoding: 3 ticks of transversal CNOTs, r SE rounds each
4. Fold-transversal S on W1-W7 + S_dag on W0 (noisy, in one unitary block)
5. r SE rounds
6. Measure all data qubits in X basis

**Expected**: 4 observables (W patches only), noiseless = 0, 1 target + 3 post-select

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import stim

from eval.logical_circuit_benchmark.distillation.tg_7to1.TG_distillation_no_inject import (
    build_no_inject_circuit,
    analyze_observables,
    run_simulation,
)
from src.simulation.observable_analysis import build_obs_patch_matrix, identify_distillation_observables
from src.noise.config import NoiseConfig
from src.noise.injector import NoiseInjector

## 1. Build circuit (d=3)

In [2]:
d = 3
r = 1
circuit, info, system = build_no_inject_circuit(d, rounds=d, r=r)

print(f"d={d}, rounds={d}, r={r}")
print(f"  qubits:      {info['num_qubits']}")
print(f"  detectors:   {info['num_detectors']}")
print(f"  observables: {info['num_observables']}")

  Init SE (3 rounds)...
Applying first round of syndrome extraction...
Applying rest rounds of syndrome extraction...
  CNOT tick 1 + SE (1 rounds)...
Applying first round of syndrome extraction...
  CNOT tick 2 + SE (1 rounds)...
Applying first round of syndrome extraction...
  CNOT tick 3 + SE (1 rounds)...
Applying first round of syndrome extraction...
  Transversal S (W1-W7) + S_dag (W0) + SE (1 rounds)...
Applying first round of syndrome extraction...
d=3, rounds=3, r=1
  qubits:      200
  detectors:   672
  observables: 4


## 2. Noiseless check

In [3]:
dets, obs = circuit.compile_detector_sampler().sample(shots=100, separate_observables=True)
print(f"Detector triggers: {np.sum(dets)} (expected 0)")
print(f"Observable triggers: {np.sum(obs)} (expected 0)")
print(f"Noiseless check: {'OK' if not np.any(dets) and not np.any(obs) else 'FAIL'}")

Detector triggers: 0 (expected 0)
Observable triggers: 0 (expected 0)
Noiseless check: OK


## 3. Observable analysis

The raw observables (L0..L3) are linear combinations of X measurements across W patches.
GF(2) elimination targeting W0 identifies:
- **Target**: the observable isolating W0 (distilled output)
- **Post-select**: syndrome checks (should be +1 in no-error case)

In [4]:
matrix, patch_names = build_obs_patch_matrix(circuit, system)
T, target_obs, ps_obs = identify_distillation_observables(matrix, patch_names, ['W0'])

w_names = ['W0','W1','W2','W3','W4','W5','W6','W7']
w_cols = [patch_names.index(n) for n in w_names]

print("Raw obs-to-patch matrix (W columns):")
print("     " + " ".join(f"{n:>5}" for n in w_names))
for i, row in enumerate(matrix):
    vals = " ".join(f"{'1' if row[c] else '0':>5}" for c in w_cols)
    print(f"  L{i}: {vals}")

M_new = (T @ matrix) % 2
print(f"\nAfter GF(2) elimination (target=W0):")
print("     " + " ".join(f"{n:>5}" for n in w_names))
for i, row in enumerate(M_new):
    vals = " ".join(f"{'1' if row[c] else '0':>5}" for c in w_cols)
    label = 'TARGET' if i in target_obs else 'PS'
    print(f"  L{i}': {vals}  [{label}]")

print(f"\nTarget obs indices: {target_obs}")
print(f"Post-select obs indices: {ps_obs}")
print(f"T matrix:\n{T}")

Raw obs-to-patch matrix (W columns):
        W0    W1    W2    W3    W4    W5    W6    W7
  L0:     1     0     1     0     1     0     1     0
  L1:     1     1     0     0     1     1     0     0
  L2:     1     1     1     1     0     0     0     0
  L3:     1     1     1     1     1     1     1     1

After GF(2) elimination (target=W0):
        W0    W1    W2    W3    W4    W5    W6    W7
  L0':     1     0     1     0     1     0     1     0  [TARGET]
  L1':     0     1     1     0     0     1     1     0  [PS]
  L2':     0     1     0     1     1     0     1     0  [PS]
  L3':     0     1     0     1     0     1     0     1  [PS]

Target obs indices: [0]
Post-select obs indices: [1, 2, 3]
T matrix:
[[1 0 0 0]
 [1 1 0 0]
 [1 0 1 0]
 [1 0 0 1]]


## 4. Verify PS observables: compare to theoretical Steane X stabilizers

For [[7,1,3]] Steane code, the X-type parity check matrix H_X has 3 rows.
Check if our 3 PS observables match the Steane H_X (possibly with qubit relabeling).

In [5]:
# PS observable matrix restricted to W1-W7 columns
ps_mat = np.array([[1 if M_new[i, patch_names.index(f'W{j}')] else 0
                    for j in range(1, 8)]
                   for i in ps_obs], dtype=int)

print("PS observables (W1-W7 columns):")
print("     " + " ".join(f"W{j:1d}" for j in range(1, 8)))
for i, row in enumerate(ps_mat):
    print(f"  PS{i}: " + " ".join(str(v) for v in row))

# Standard Steane [[7,1,3]] H_X (one possible form)
H_X_steane = np.array([
    [1,0,0,1,0,1,1],
    [0,1,0,1,1,0,1],
    [0,0,1,0,1,1,1],
], dtype=int)
print("\nStandard Steane H_X:")
print("     " + " ".join(f"W{j:1d}" for j in range(1, 8)))
for i, row in enumerate(H_X_steane):
    print(f"  HX{i}: " + " ".join(str(v) for v in row))

# Check rank
rank = np.linalg.matrix_rank(ps_mat % 2)
print(f"\nPS matrix rank over GF(2): {rank} (expected 3 for [[7,1,3]])")

# Check if they span the same GF(2) row space (i.e., same code)
combined = np.vstack([ps_mat, H_X_steane]) % 2
combined_rank = int(np.linalg.matrix_rank(combined))
print(f"Rank of [PS | Steane H_X] combined: {combined_rank}")
print("Same GF(2) row space (same code)?:", combined_rank == rank)

PS observables (W1-W7 columns):
     W1 W2 W3 W4 W5 W6 W7
  PS0: 1 1 0 0 1 1 0
  PS1: 1 0 1 1 0 1 0
  PS2: 1 0 1 0 1 0 1

Standard Steane H_X:
     W1 W2 W3 W4 W5 W6 W7
  HX0: 1 0 0 1 0 1 1
  HX1: 0 1 0 1 1 0 1
  HX2: 0 0 1 0 1 1 1

PS matrix rank over GF(2): 3 (expected 3 for [[7,1,3]])
Rank of [PS | Steane H_X] combined: 6
Same GF(2) row space (same code)?: False


## 5. DEM analysis: dangerous errors that escape post-selection

**Key**: post-selection is applied in the **transformed frame** (after T), NOT the raw observable frame.

For this circuit, T = [[1,0,0,0],[1,1,0,0],[1,0,1,0],[1,0,0,1]], so:
- L0' = L0  (target — contains W0)
- L1' = L0 ⊕ L1 (PS)
- L2' = L0 ⊕ L2 (PS)
- L3' = L0 ⊕ L3 (PS)

An error that flips **all 4 raw** observables [0,1,2,3]:
  → transformed = [1,0,0,0] → **escapes PS** (dangerous)

An error that flips **only raw L0**:
  → transformed = [1,1,1,1] → **fails PS** (safe)

**Physically**: W0 appears in all 4 raw observables because it is the logical output qubit.
Any logical X error on W0 always flips [0,1,2,3] in the raw frame → trans=[1,0,0,0].
This is by design: we are *measuring* W0's LER, not post-selecting it away.

The DECODER (BPOSD) is responsible for correcting these errors (they all have n_dets≥1).

In [6]:
p = 1e-3
noise_config = NoiseConfig(p_1q=p, p_2q=p, p_meas=p, p_reset=p, p_idle=p)
injector = NoiseInjector.from_circuit_level(noise_config, list(range(circuit.num_qubits)))
noisy = injector.inject_noise(circuit)

dem = noisy.detector_error_model(approximate_disjoint_errors=True)
print(f"DEM: {dem.num_errors} errors, {dem.num_detectors} detectors, {dem.num_observables} observables")

n_obs = circuit.num_observables

# Correct analysis: check escape in TRANSFORMED frame, not raw frame.
# An error is dangerous iff:
#   trans_flip = (T @ raw_flip) % 2  has  trans[target_obs]=1  AND  trans[ps_obs]=0
dangerous = []
raw_pattern_counts = {}

for instruction in dem:
    if not (isinstance(instruction, stim.DemInstruction) and instruction.type == 'error'):
        continue
    targets = instruction.targets_copy()
    raw_flipped_ids = [t.val for t in targets if t.is_logical_observable_id()]
    flipped_dets    = [t.val for t in targets if t.is_relative_detector_id()]

    raw_flip = np.zeros(n_obs, dtype=int)
    for idx in raw_flipped_ids:
        raw_flip[idx] = 1

    # Apply T to get transformed observables
    trans_flip = (T @ raw_flip) % 2

    target_flipped = any(trans_flip[i] == 1 for i in target_obs)
    ps_all_zero    = all(trans_flip[i] == 0 for i in ps_obs)

    if target_flipped and ps_all_zero:
        key = tuple(sorted(raw_flipped_ids))
        raw_pattern_counts[key] = raw_pattern_counts.get(key, 0) + 1
        dangerous.append({
            'prob':   instruction.args_copy()[0],
            'n_dets': len(flipped_dets),
            'raw_obs': raw_flipped_ids,
            'trans':   trans_flip.tolist(),
        })

print(f"\nDangerous errors (escape PS in transformed frame): {len(dangerous)}")

# Distribution by n_dets
by_ndets = {}
for e in dangerous:
    by_ndets[e['n_dets']] = by_ndets.get(e['n_dets'], 0) + 1
print("  n_dets distribution:", dict(sorted(by_ndets.items())))

print("\nRaw observable flip patterns of dangerous errors:")
for k, cnt in sorted(raw_pattern_counts.items(), key=lambda x: -x[1]):
    rv = np.zeros(n_obs, dtype=int)
    for idx in k: rv[idx] = 1
    tv = (T @ rv) % 2
    print(f"  raw={list(k)}  trans={tv.tolist()}  count={cnt}")

# Note: n_dets=0 would be truly undetectable (irreducible floor).
# n_dets>=1 means the decoder can see the syndrome and should correct.
n_undetectable = sum(1 for e in dangerous if e['n_dets'] == 0)
print(f"\nUndetectable (n_dets=0, irreducible floor): {n_undetectable}")
print(f"Detectable (n_dets>=1, decoder can correct): {len(dangerous) - n_undetectable}")

DEM: 11324 errors, 672 detectors, 4 observables



Dangerous errors (escape PS in transformed frame): 227
  n_dets distribution: {1: 27, 2: 47, 3: 75, 4: 34, 5: 35, 6: 1, 7: 1, 9: 7}

Raw observable flip patterns of dangerous errors:
  raw=[0, 1, 2, 3]  trans=[1, 0, 0, 0]  count=227

Undetectable (n_dets=0, irreducible floor): 0
Detectable (n_dets>=1, decoder can correct): 227


## 6. Quick LER check (CPU BPOSD, small sample)

In [ ]:
import time, math
from src.simulation.decoder_backend.registry import get_decoder
from src.simulation.observable_analysis import transform_observables

p = 1e-3
batch = 10_000
max_errors = 20

decoder = get_decoder('bposd', backend='cpu')
dem_bposd = noisy.detector_error_model(approximate_disjoint_errors=True)
compiled = decoder.compile_decoder_for_dem(dem=dem_bposd)
sampler = noisy.compile_detector_sampler()

total_shots = kept_shots = errors = 0
t0 = time.perf_counter()

while errors < max_errors and total_shots < 200_000:
    dts, obs = sampler.sample(shots=batch, separate_observables=True)
    total_shots += batch
    obs_t = transform_observables(obs, T)
    mask = np.all(obs_t[:, ps_obs] == 0, axis=1)
    if not np.any(mask):
        continue
    dts_k = dts[mask]
    obs_k = obs_t[mask]
    kept_shots += dts_k.shape[0]
    n_det = dem_bposd.num_detectors
    n_det_bytes = math.ceil(n_det / 8)
    dts_packed = np.packbits(dts_k, axis=1, bitorder='little')[:, :n_det_bytes]
    pred_packed = compiled.decode_shots_bit_packed(bit_packed_detection_event_data=dts_packed)
    pred_unp = np.unpackbits(pred_packed, axis=1, bitorder='little')[:, :circuit.num_observables]
    pred_t = transform_observables(pred_unp, T)
    corr = obs_k[:, target_obs] ^ pred_t[:, target_obs]
    errors += int(np.sum(np.any(corr, axis=1)))

elapsed = time.perf_counter() - t0
ler = errors / kept_shots if kept_shots > 0 else 0
ps_rate = kept_shots / total_shots if total_shots > 0 else 0
print(f"d={d}, p={p:.0e}")
print(f"  shots={total_shots:,}, kept={kept_shots:,} ({ps_rate*100:.1f}%), errors={errors}")
print(f"  LER = {ler:.3e}  ({elapsed:.1f}s)")